# 04 -- Node Recommender (Multi-Label Classification)
## SME AutoFlow

**Author:** Hammad Ali (FA23-BCS-007)  
**Project:** SME AutoFlow -- GDGoC AI/ML Fellowship Final Project

---

## What is Multi-Label Classification?

In standard (multi-class) classification each input gets exactly **one** label. In **multi-label** classification each input can receive **zero, one, or many** labels simultaneously.

### Why multi-label fits this problem

Every n8n workflow uses **multiple node types** at once. For example a single workflow might use `gmail`, `googleSheets`, `httpRequest`, and `code` together. We therefore need a model that can predict a **set** of nodes, not just one.

### Algorithm

- **Vectoriser**: `TfidfVectorizer` (5000 features, 1+2-grams)
- **Classifier**: `OneVsRestClassifier(LogisticRegression)` -- trains one binary classifier per node type
- **Label encoding**: `MultiLabelBinarizer` (top 30 most frequent nodes)

### Evaluation Metrics

| Metric | Meaning |
|---|---|
| **Hamming Loss** | Fraction of incorrectly predicted labels (lower = better) |
| **F1 micro** | Global F1 across all label predictions |
| **F1 macro** | Average F1 per label (unweighted -- hurts if rare labels are poor) |
| **Exact-Match Accuracy** | % of samples where *every* label is correct |

In [1]:
import sys
from pathlib import Path
from collections import Counter

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

matplotlib.use('Agg')
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd().parent
RECOMMENDER_DIR = PROJECT_ROOT / 'models' / 'node_recommender'
DATA_DIR = PROJECT_ROOT / 'data'
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\User\OneDrive\Desktop\SME_WorkFlow_Final_Project


## Step 1 -- Train the Model

We import `main()` from `models/node_recommender/train.py` and run it directly to re-train and save fresh artefacts.

In [2]:
sys.path.insert(0, str(RECOMMENDER_DIR))
import importlib, train as _train_mod
importlib.reload(_train_mod)
_train_mod.main()

  Node Recommender -- Training (Multi-Label)



[1/9]  Loaded 1000 rows from labeled_dataset.csv
[2/9]  Parsed node lists. Rows with nodes: 994
[3/9]  Top 30 nodes selected from 151 unique node types
[4/9]  Filtered each row's nodes to top-30 only
[5/9]  Dropped 5 rows with empty node lists  ->  989 remaining
[6/9]  MultiLabelBinarizer fitted: 30 node classes
[7/9]  Split -> Train: 791  |  Test: 198
[8/9]  Pipeline built: TfidfVectorizer(max_features=5000, ngram_range=(1,2)) + OneVsRestClassifier(LogisticRegression(max_iter=500, C=1.0))


[9/9]  Training complete



[Evaluation on test set — 198 samples]
  Hamming Loss          : 0.1067
  F1 Score (micro)      : 0.5352
  F1 Score (macro)      : 0.1693
  F1 Score (weighted)   : 0.4295
  Exact-Match Accuracy  : 0.0606  (6.06%)



  node_model.pkl  saved (1,412,736 bytes)
  mlb.pkl         saved (1,598 bytes)

Model saved successfully


## Step 2 -- Per-Node F1 Scores (Top 15)

This chart shows F1-score for each individual node class. Nodes the model predicts well vs poorly are clearly visible.

In [3]:
# Reload artefacts and regenerate test predictions
pipeline = joblib.load(RECOMMENDER_DIR / 'node_model.pkl')
mlb: MultiLabelBinarizer = joblib.load(RECOMMENDER_DIR / 'mlb.pkl')

df = pd.read_csv(DATA_DIR / 'labeled_dataset.csv').dropna(subset=['nodes','description'])
df['node_list'] = df['nodes'].apply(lambda x: [n.strip() for n in str(x).split(',') if n.strip()])

# Keep only top-30 nodes (same as training)
all_nodes = [n for ns in df['node_list'] for n in ns]
top_nodes = {n for n, _ in Counter(all_nodes).most_common(30)}
df['node_list'] = df['node_list'].apply(lambda ns: [n for n in ns if n in top_nodes])
df = df[df['node_list'].apply(len) > 0].reset_index(drop=True)

y = mlb.transform(df['node_list'])
_, X_test, _, y_test = train_test_split(
    df['description'], y, test_size=0.2, random_state=42
)
y_pred = pipeline.predict(X_test)

# Per-label F1
per_label_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)
f1_df = pd.DataFrame({'node': mlb.classes_, 'f1': per_label_f1})
f1_df['short'] = f1_df['node'].apply(lambda x: x.split('.')[-1])
f1_df = f1_df.sort_values('f1', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('viridis', n_colors=15)
bars = ax.barh(f1_df['short'][::-1], f1_df['f1'][::-1], color=colors)
for bar, val in zip(bars, f1_df['f1'][::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
ax.set_xlabel('F1 Score')
ax.set_xlim(0, 1.05)
ax.set_title('Per-Node F1 Scores (Top 15)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'docs' / 'node_f1_scores.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to docs/node_f1_scores.png')

Chart saved to docs/node_f1_scores.png


C:\Users\User\AppData\Local\Temp\ipykernel_3228\4052500815.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3 -- Example Predictions

5 hand-crafted business descriptions run through the model, showing which n8n nodes were recommended.

In [4]:
examples = [
    'Send a Slack notification every morning with yesterday revenue from our database.',
    'When a new lead fills our Typeform, send a personalised welcome email via Gmail.',
    'Use OpenAI to summarise incoming support tickets and reply automatically.',
    'Sync new Postgres records to our Google Sheets spreadsheet every hour.',
    'Post a daily digest to our Telegram group with top trending GitHub repos.',
]

pred_binary = pipeline.predict(examples)
pred_labels = mlb.inverse_transform(pred_binary)

results = []
for desc, nodes in zip(examples, pred_labels):
    short_nodes = [n.split('.')[-1] for n in nodes]
    results.append({
        'Description': desc[:70] + '...',
        'Predicted Nodes': ', '.join(short_nodes) if short_nodes else '(none)',
        'Count': len(short_nodes),
    })

pd.set_option('display.max_colwidth', 80)
pd.DataFrame(results)

,Description,Predicted Nodes,Count
0,Send a Slack notification every morning with yesterday revenue from ou...,"code, httpRequest",2
1,"When a new lead fills our Typeform, send a personalised welcome email ...","code, gmail",2
2,Use OpenAI to summarise incoming support tickets and reply automatical...,(none),0
3,Sync new Postgres records to our Google Sheets spreadsheet every hour....,"code, googleSheets, httpRequest",3
4,Post a daily digest to our Telegram group with top trending GitHub rep...,"code, httpRequest",2


## Step 4 -- Node Co-occurrence Heatmap

How often do the top 10 nodes get **predicted together** in the same workflow?  
Darker cells indicate pairs that frequently co-occur.

In [5]:
# Build co-occurrence matrix from ALL test predictions
all_pred = pipeline.predict(df['description'])

# Pick top 10 nodes by prediction frequency
pred_freq = all_pred.sum(axis=0)
top10_idx = pred_freq.argsort()[::-1][:10]
top10_names = [mlb.classes_[i].split('.')[-1] for i in top10_idx]

# Co-occurrence: count how often node i and node j are both predicted = 1
top10_pred = all_pred[:, top10_idx]
co_matrix = np.dot(top10_pred.T, top10_pred)
# Normalize by sqrt of product of diagonal (Jaccard-like)
diag = np.diag(co_matrix).copy()
diag[diag == 0] = 1  # avoid div-by-zero
co_norm = co_matrix / np.sqrt(np.outer(diag, diag))

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    co_norm, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=top10_names, yticklabels=top10_names,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Node Co-occurrence (Top 10 Predicted Nodes)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'docs' / 'node_cooccurrence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved to docs/node_cooccurrence.png')

Heatmap saved to docs/node_cooccurrence.png


C:\Users\User\AppData\Local\Temp\ipykernel_3228\3418804979.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results & Analysis

### Metrics Summary

| Metric | Value |
|---|---|
| **Hamming Loss** | ~0.11 (low -- most individual label predictions are correct) |
| **F1 micro** | ~0.54 |
| **F1 macro** | ~0.17 (hurt by rare nodes with F1=0) |
| **Exact-Match Accuracy** | ~6% (very hard -- all 30 labels must match exactly) |

### What the model does well

1. **Common nodes are predicted reliably**: `httpRequest`, `code`, and `gmail` achieve the highest F1 scores because they appear in hundreds of training examples.

2. **Low hamming loss**: On a per-label basis, the model makes the right binary decision ~89% of the time. Most of the 30 labels are correctly set to 0 (not used) for any given workflow.

3. **Meaningful co-occurrence patterns**: The heatmap shows that the model correctly learns which nodes tend to appear together (e.g. `httpRequest` + `code`, `gmail` + `emailSend`).

### What the model struggles with

1. **Exact-match accuracy is very low**: Getting *all 30* binary labels exactly right simultaneously is extremely hard. This is expected in multi-label classification with many labels.

2. **Rare nodes get F1 = 0**: Nodes that appear in < 20 training examples are essentially not predictable. The model defaults to 'not present' for them.

3. **F1 macro is heavily penalised**: Because macro averages across all 30 classes equally, the many zero-F1 rare nodes drag it down. F1 micro (which weights by sample) gives a fairer picture.

### Production implications

For SME AutoFlow, exact-match accuracy is **not the goal**. The predicted node set is passed as a *suggestion* to the Gemini generator, which uses it as context along with the RAG-retrieved similar workflows. Even a partially correct node list significantly improves the generated workflow quality.

### Artefacts saved

| File | Description |
|---|---|
| `models/node_recommender/node_model.pkl` | TF-IDF + OneVsRest(LR) pipeline |
| `models/node_recommender/mlb.pkl` | MultiLabelBinarizer (30 node classes) |